In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Backpropagation Through Time

Our experiments with the RNN language model in that section left
us with two striking empirical findings: without gradient clipping, most
training runs destabilized; with clipping, training converged reliably, yet
the learned network used only the last few tokens of its 32-token context
window. Both phenomena share the exact same mathematical root. Because a
recurrent network applies the same weight matrix at every time step, any
gradient propagating backward through a sequence is multiplied by a state
Jacobian at each step. Over many steps these factors compound—much like
compound interest—causing the gradient to grow or shrink geometrically.

Even a scalar recurrence $h_t = w_\textrm{h} h_{t-1} + x_t$ shows why:
unrolling gives $h_t = \sum_{s \leq t} w_\textrm{h}^{t-s} x_s$, so
$\partial h_t / \partial h_{t-k} = w_\textrm{h}^k$. Over forty steps,
$w_\textrm{h} = 0.9$ shrinks a gradient to $0.9^{40} \approx 0.015$, whereas
$w_\textrm{h} = 1.1$ inflates it to $1.1^{40} \approx 45$. Exploding gradients
destabilize optimization immediately, forcing us to clip updates; vanishing
gradients are quieter yet more insidious, attenuating error signals from
distant tokens until the model cannot learn long-range dependencies.

Computing gradients in an RNN requires no new differentiation algorithm: we
unroll the recurrence across time into a deep feedforward network with shared
weights and apply standard backpropagation (that section), a
procedure known as *backpropagation through time* (BPTT) [@Werbos.1990].
Below, we derive the backward recursion directly in matrix form, measure how
far gradients actually travel inside our trained RNN from
that section, and examine truncated backpropagation through time.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Vanishing and Exploding Gradients

Consider an RNN unrolled across $T$ steps with shared recurrent parameters and
average sequence loss $L = \frac{1}{T}\sum_{t=1}^T l(\mathbf{y}_t,
\mathbf{o}_t)$. Backpropagation handles weight sharing by summing the local
gradient contributions from every time step where a weight matrix appears. To
see the structure of the backward pass transparently, take the RNN of
that section and temporarily omit biases and the $\tanh$ nonlinearity (we
restore $\tanh$ immediately afterward). For a minibatch of $b$ sequences with
step inputs $\mathbf{X}_t \in \mathbb{R}^{b\times d}$, the linear RNN computes

$$
\mathbf{H}_t = \mathbf{X}_t \mathbf{W}_\textrm{xh} +
\mathbf{H}_{t-1}\mathbf{W}_\textrm{hh}, \qquad
\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_\textrm{hq},
$$

with $\mathbf{W}_\textrm{xh}\in\mathbb{R}^{d\times h}$,
$\mathbf{W}_\textrm{hh}\in\mathbb{R}^{h\times h}$, and
$\mathbf{W}_\textrm{hq}\in\mathbb{R}^{h\times q}$. the figure
depicts three unrolled time steps of this network, showing the forward pass in
grey and the backward pass in red.

![Forward pass (grey) and backward pass (red) of a linear RNN unrolled over three time steps. The gradient that reaches each hidden state except the last is the sum of two red arrows: one from its own output, through $\mathbf{W}_\textrm{hq}^\top$, and one from the next state, through $\mathbf{W}_\textrm{hh}^\top$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-rnn-bptt.svg)

Let $\mathbf{G}_t = \partial L/\partial \mathbf{H}_t$ denote the gradient of
the total loss with respect to $\mathbf{H}_t$. Because $\mathbf{H}_t$ feeds
into both its immediate output $\mathbf{O}_t$ and the next hidden state
$\mathbf{H}_{t+1}$, and backpropagating through a right-multiplication
multiplies by the transposed weight matrix (that section),
$\mathbf{G}_t$ obeys the backward recursion

$$
\mathbf{G}_t = \frac{\partial L}{\partial \mathbf{O}_t}\mathbf{W}_\textrm{hq}^\top +
\mathbf{G}_{t+1}\mathbf{W}_\textrm{hh}^\top, \qquad \mathbf{G}_{T+1} = \mathbf{0},
$$

corresponding to the two incoming red arrows at each hidden state in
the figure. Unrolling the equation backward
from $\mathbf{G}_T$ down to $\mathbf{G}_t$ yields

$$
\mathbf{G}_t = \sum_{j=t}^{T} \frac{\partial L}{\partial \mathbf{O}_j}\,
\mathbf{W}_\textrm{hq}^\top\big(\mathbf{W}_\textrm{hh}^\top\big)^{j-t}.
$$

Thus, the loss at future step $j$ communicates back to the state $k = j - t$
steps earlier through the $k^\textrm{th}$ power of
$\mathbf{W}_\textrm{hh}^\top$ (for a general nonlinear transition
$\mathbf{H}_t = g(\mathbf{H}_{t-1}, \mathbf{X}_t, \mathbf{W}_\textrm{hh})$,
this matrix power is replaced by the product of the $j - t$ intermediate state
Jacobians $\prod_{i=t+1}^j \partial \mathbf{H}_i / \partial \mathbf{H}_{i-1}$).
Once $\mathbf{G}_1, \ldots, \mathbf{G}_T$ are computed in a single backward
sweep, the parameter gradients simply sum one outer product per step:

$$
\frac{\partial L}{\partial \mathbf{W}_\textrm{hh}} = \sum_{t=1}^{T}
\mathbf{H}_{t-1}^\top \mathbf{G}_t, \qquad
\frac{\partial L}{\partial \mathbf{W}_\textrm{xh}} = \sum_{t=1}^{T}
\mathbf{X}_t^\top \mathbf{G}_t.
$$

As we saw in that section and
that section, raising a matrix to the $k^\textrm{th}$
power scales each eigenvector by $\lambda^k$. Letting $\rho$ denote the
*spectral radius* (largest $|\lambda|$) of $\mathbf{W}_\textrm{hh}$, the norm
of $(\mathbf{W}_\textrm{hh}^\top)^k$ scales asymptotically as $\rho^k$: when
$\rho < 1$, contributions from distant steps vanish exponentially; when
$\rho > 1$, they explode exponentially. Reintroducing the $\tanh$ activation
multiplies each step's backward gradient entrywise by its local derivative
$1 - \mathbf{H}_t^2 \in (0, 1]$ before multiplying by
$\mathbf{W}_\textrm{hh}^\top$. Because $1 - h^2 \leq 1$, $\tanh$ never
amplifies a gradient and pulls strongly toward vanishing whenever hidden units
saturate near $\pm 1$, while still permitting explosion if
$\mathbf{W}_\textrm{hh}$ is large and activations stay near zero.

Let us measure what actually happens inside our trained language model from
that section ($d = 512$, $h = 128$, with dropout).

In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)

In [ ]:
rnn = d2l.RNNScratch(num_inputs=512, num_hiddens=128)
model = d2l.RNNLMScratch(rnn, vocab_size=len(data.vocab), lr=4, dropout=0.5,
                         emb_dropout=0.1)

In [ ]:
trainer = d2l.Trainer(max_epochs=10, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

We run the trained model forward over 256 validation segments of length
$L = 64$—twice its training window—recording all hidden states and the
final-step logits.

In [ ]:
L = 64
val = d2l.numpy(data.X[data.num_train:data.num_train + data.num_val, 0])
starts = np.random.default_rng().choice(len(val) - L, 256, replace=False)
seqs = np.stack([val[s:s + L] for s in starts])  # (256, L) token ids
targets = val[starts + L]                        # The token after each stretch

In [ ]:
states, _ = model.rnn(model.embedding(jnp.array(seqs)))
logits = model.output_layer(states[-1:])[:, 0]  # The last step only
H, logits = d2l.numpy(jnp.stack(states, 0)), d2l.numpy(logits)
W_hh, W_hq = d2l.numpy(model.rnn.W_hh[...]), d2l.numpy(model.W_hq[...])

Starting from the final-step loss gradient $\partial L_L / \partial
\mathbf{O}_L = \textrm{softmax}(\mathbf{O}_L) - \mathbf{Y}_L$, we step backward
through time via $\mathbf{G}_{t-1} = (\mathbf{G}_t \odot (1 -
\mathbf{H}_t^2))\mathbf{W}_\textrm{hh}^\top$ and plot the geometric mean of
the relative gradient norm $\|\mathbf{G}_{L-k}\| / \|\mathbf{G}_L\|$ as a
function of the lag $k$.

In [ ]:
P = np.exp(logits - logits.max(1, keepdims=True))
P /= P.sum(1, keepdims=True)
P[np.arange(len(targets)), targets] -= 1  # dL/dO: softmax minus one-hot
G = (P @ W_hq.T).astype(np.float64)       # dL/dH at the last step
norms = []
for t in range(L - 1, -1, -1):            # Backward through time
    norms.append(np.linalg.norm(G, axis=1))
    G = (G * (1 - H[t] ** 2)) @ W_hh.T    # Through the tanh, then W_hh
rel = np.exp(np.log(np.array(norms) / norms[0]).mean(1))
print(f'spectral radius of W_hh: {np.abs(np.linalg.eigvals(W_hh)).max():.2f}; '
      f'mean tanh derivative: {(1 - H ** 2).mean():.2f}')
print('relative gradient norm at lags 1, 2, 4, 8, 16, 32:',
      [float(f'{rel[k]:.2g}') for k in (1, 2, 4, 8, 16, 32)])
d2l.plot(np.arange(L), rel, 'lag $k$', 'relative gradient norm',
         yscale='log', figsize=(4.5, 3))

Even though the trained $\mathbf{W}_\textrm{hh}$ has a spectral radius near
$1.8$—which would explode in a purely linear recurrence—the actual gradient
vanishes rapidly: it drops below 1% of its initial norm within roughly eight
steps and shrinks to a few billionths across 32 tokens. Because the $\tanh$
derivative averages roughly $0.27$, each backward step retains only about
$1.8 \times 0.27 \approx 0.5$ of the gradient norm. This directly explains our
finding in that section that per-position validation loss
flattens out after roughly three tokens of context.

### Remedies for Exploding and Vanishing Gradients

Exploding and vanishing gradients demand fundamentally different remedies.
Because an exploding gradient threatens to corrupt the parameters in a single
oversized update, global gradient clipping (the equation) provides
a simple safeguard—yet multiplying the entire gradient sum
the equation by a scalar never changes the *relative* weight
of distant versus recent lags. Overcoming vanishing gradients therefore
requires changing the recurrence itself so that state Jacobians stay near unity
along memory directions: either via *gated* recurrent cells (such as LSTMs and
GRUs) that create an additive memory path controlled by learned gates, or via
*structured linear recurrences* and state space models that remove the state
nonlinearity and constrain eigenvalues near the unit circle. Both families are
developed in that section.

## Truncated Backpropagation Through Time

Evaluating the equation across an entire corpus of $T$
tokens requires retaining all $T$ forward hidden states in memory and
traversing all $T$ steps for a single parameter update. *Truncated
backpropagation through time* caps both memory and compute per update by
slicing long streams into fixed-length segments of length $T_0$ and halting
backpropagation at each segment boundary (the figure).

![Full and truncated backpropagation through time over a token sequence. Both run the same forward pass (grey). Full backpropagation sends the gradient (red) back along the entire sequence. Truncation cuts the sequence into segments of four tokens and detaches the state at each boundary: its value crosses into the next segment, but no gradient flows back across.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-rnn-truncated-bptt.svg)

Whether each window resets $\mathbf{H}_0 = \mathbf{0}$ (as in
that section) or carries the previous segment's final state
forward while *detaching* it from the computational graph (as in
the figure), no gradient ever crosses a segment boundary,
imposing a hard learning horizon of at most $T_0$ steps [@Jaeger.2002]. In
a vanilla RNN, however, vanishing gradients already choke off the backward
signal after roughly eight steps—well inside our 32-token window—so lengthening
the window alone cannot extend the model's effective memory.

## Summary

Backpropagation through time applies ordinary backpropagation to an unrolled
RNN, propagating loss sensitivities backward via
the equation and summing parameter gradients across time
steps (the equation). Because a gradient traveling back $k$
steps multiplies $k$ state Jacobians—reducing in a linear RNN to
$(\mathbf{W}_\textrm{hh}^\top)^k \sim \rho^k$—gradients explode when $\rho > 1$
and vanish when $\rho < 1$. Including $\tanh$ multiplies each step by
$1 - \mathbf{H}_t^2 \in (0, 1]$, causing gradients in our trained RNN to fall
below 1% within roughly eight steps despite $\rho \approx 1.8$. Gradient
clipping contains exploding spikes, truncated BPTT bounds memory and compute
per update, and architectural innovations (that section) are
required to preserve gradients over long horizons.

## Exercises

1. **Backward pass by hand.** One step of
   `RNNScratch` (that section) computes
   $$\mathbf{H}_t = \tanh(\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}} + \mathbf{b}_\textrm{h}).$$
   Let $\mathbf{G}_t = \partial L / \partial \mathbf{H}_t$ be the gradient
   arriving from later steps and from the output at step $t$.
    1. Derive the contributions of this step to
       $\partial L / \partial \mathbf{W}_{\textrm{xh}}$,
       $\partial L / \partial \mathbf{W}_{\textrm{hh}}$,
       $\partial L / \partial \mathbf{b}_\textrm{h}$, and
       $\partial L / \partial \mathbf{H}_{t-1}$ as functions of
       $\mathbf{G}_t$, $\mathbf{X}_t$, $\mathbf{H}_t$, $\mathbf{H}_{t-1}$,
       and $\mathbf{W}_{\textrm{hh}}$.
    1. Adding the output contribution at step $t-1$ to the contribution
       to $\partial L / \partial \mathbf{H}_{t-1}$ gives a recursion for
       $\mathbf{G}_{t-1}$. Show that it reduces to
       the equation when the $\tanh$ and the bias
       are dropped, and name the factor that the $\tanh$ adds at every step.
       Explain also how the parameter gradients accumulate over the $T$
       steps of a window, and compare with the equation.

    *Adapted from CMU 11-785,
    [Homework 3 Part 1](https://deeplearning.cs.cmu.edu/F22/document/homework/HW3/HW3P1_F22.pdf).*

1. **Power iteration and alignment.** Let
   $\mathbf{M} \in \mathbb{R}^{h \times h}$ be symmetric with eigenvalues
   $\lambda_i$ and eigenvectors $\mathbf{v}_i$ for $i = 1, \ldots, h$,
   ordered so that $|\lambda_i| \geq |\lambda_{i+1}|$.
    1. Show that $\mathbf{M}^k$ has eigenvalues $\lambda_i^k$ with the
       same eigenvectors.
    1. Show that for a random vector $\mathbf{x} \in \mathbb{R}^h$ with
       independent standard normal entries, $\mathbf{M}^k \mathbf{x}$
       becomes aligned with $\mathbf{v}_1$ as $k$ grows. State the
       condition on the eigenvalues that this requires and the sense in
       which the alignment holds.
    1. What does this imply for the gradient the equation
       of an RNN, beyond the change in its norm?

1. [code] **Backward pass in code.** ● Train `d2l.RNNLMScratch` on
   `d2l.TimeMachine` as in that section. Using the
   single-step gradients and the recursion of problem 1,
   implement the backward pass of its `RNNScratch` cell over a full
   window with tensor operations only, starting from the gradient of the
   loss with respect to the
   logits (that section). Run it on one training
   minibatch, and compare your gradients of `W_xh`, `W_hh`, and `b_h`
   with those computed by automatic differentiation.

    *Adapted from CMU 11-785,
    [Homework 3 Part 1](https://deeplearning.cs.cmu.edu/F22/document/homework/HW3/HW3P1_F22.pdf).*

1. [code] **Nonlinear Jacobians.** ● Draw a random symmetric
   $100 \times 100$ matrix, rescale it to spectral radius
   $\rho \in \{0.9, 1.0, 1.1\}$, and plot the norm of its $k^\textrm{th}$
   power $\|\mathbf{J}^k\|$ against $k$ on a logarithmic axis: three
   straight lines, as the eigenvalues predict. Then leave the symmetric
   linear case.
    1. Replace the symmetric matrix by a general, nonsymmetric
       $\mathbf{W}_\textrm{hh}$ rescaled to each radius, and plot
       $\|\mathbf{J}^k\|$ against $k$. Report the largest value of
       $\|\mathbf{J}^k\|$ at $\rho = 0.9$ and the lag at which it occurs.
       Explain why the ratio $\|\mathbf{J}^k\| / \rho^k$ can exceed one
       and why it stays bounded as $k$ grows (a random matrix is
       diagonalizable with probability one).
    1. Return to the symmetric $\mathbf{W}_\textrm{hh}$ and run the
       $\tanh$ RNN
       $$\mathbf{H}_t = \tanh(\mathbf{H}_{t-1}\mathbf{W}_\textrm{hh} + \mathbf{X}_t), \qquad \mathbf{H}_0 = \mathbf{0},$$
       for a single sequence ($\mathbf{H}_t \in \mathbb{R}^{1 \times h}$)
       with inputs $\mathbf{X}_t$ whose entries are drawn independently
       from $\mathcal{N}(0, s^2)$.
       Plot the norm of the product of the Jacobians
       $\partial \mathbf{H}_j / \partial \mathbf{H}_{j-1}$ along the
       trajectory for $s = 0.1$ and $s = 1$. How do the nonlinearity and
       the input scale change the growth of the norm, and why?
    1. Explain why a largest singular value of $\mathbf{W}_\textrm{hh}$
       below one is sufficient for the product to shrink, whereas a
       largest singular value above one is necessary but not sufficient
       for it to grow [@bengio1994learning; @Pascanu.Mikolov.Bengio.2013].
       Relate the argument to your $\tanh$ curves at $\rho = 0.9$ and
       $\rho = 1.1$.

1. [code] **Window length and memory horizon.** ● In the model of
   that section, the gradient of the last loss
   falls below 1% of its size within about eight steps, well inside the
   training window of 32 tokens.
    1. Before running anything: if the model is retrained with
       `num_steps=128`, do you expect this horizon to move? Argue from the
       product of state Jacobians in the equation.
    1. Retrain with `num_steps` set to 128, measure the gradient against
       the lag on stretches of 256 validation tokens, and compare the
       result with your prediction.
    1. Repeat the loss-by-position measurement of
       that section for the retrained model. Does the loss
       now keep improving beyond the third token of context?

[Discussions](https://d2l.discourse.group/t/334)